# Day 7: Deep Dive — Attention & Context
Objective: Understand how attention works inside a transformer,
what the context window is, and why it matters for LLM engineering.

In [1]:
!pip install transformers torch bertviz -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.0 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2", output_attentions=True)
model.eval()
print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded.


## 1. What is Attention?
At every step, the model looks at ALL previous tokens and decides
which ones are most relevant for predicting the next token.
This "looking back" is called attention.

Each token produces 3 vectors:
- Q (Query)  → "what am I looking for?"
- K (Key)    → "what do I contain?"
- V (Value)  → "what do I contribute?"

Attention score = softmax(Q · K^T / sqrt(d_k)) · V

In [3]:
prompt = "The cat sat on the mat because it was tired"
inputs = tokenizer(prompt, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:", tokens)
print("Token count:", len(tokens))

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

attentions = outputs.attentions
print(f"\nNumber of attention layers : {len(attentions)}")
print(f"Shape of each layer        : {attentions[0].shape}")
print("Shape means: (batch, num_heads, seq_len, seq_len)")

Tokens: ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe', 'Ġmat', 'Ġbecause', 'Ġit', 'Ġwas', 'Ġtired']
Token count: 10

Number of attention layers : 6
Shape of each layer        : torch.Size([1, 12, 10, 10])
Shape means: (batch, num_heads, seq_len, seq_len)


In [4]:
## 2. Inspecting Attention Weights
attentions[layer][batch][head][token_i][token_j]
= how much token_i attends to token_j

SyntaxError: invalid syntax (3154631738.py, line 3)

In [ ]:
# Look at layer 0, head 0
layer = 0
head = 0

attn_weights = attentions[layer][0][head].numpy()

print(f"Attention weights — Layer {layer}, Head {head}")
print(f"Shape: {attn_weights.shape} (seq_len x seq_len)\n")

# Print as a simple table
import numpy as np
np.set_printoptions(precision=3, suppress=True)

print("Tokens:", tokens)
print("\nAttention matrix (row = query token, col = key token):")
print(attn_weights)

## 3. What Token Does Each Token Attend To Most?
For each token, find which previous token it attends to most.

In [ ]:
print("=== MAX ATTENTION PER TOKEN ===\n")
for i, token in enumerate(tokens):
    attn_row = attn_weights[i]
    max_idx = attn_row.argmax()
    max_score = attn_row[max_idx]
    attended_token = tokens[max_idx]
    print(f"Token '{token}' attends most to '{attended_token}' (score: {max_score:.3f})")

## 4. Context Window
The context window = maximum number of tokens the model can
"see" at once during a forward pass.

distilgpt2 context window = 1024 tokens
GPT-4 = 128,000 tokens
Claude = 200,000 tokens

Beyond the context window, the model has NO memory of earlier text.
This is why RAG exists — to inject relevant context back in.

In [5]:
print("distilgpt2 max position embeddings:",
      model.config.max_position_embeddings)

# Show what happens as prompt gets longer
test_prompts = [
    "AI",
    "AI is transforming the world",
    "AI is transforming the world in ways we never imagined possible before",
]

print("\n=== TOKEN COUNT VS CONTEXT WINDOW ===\n")
for p in test_prompts:
    token_count = len(tokenizer.encode(p))
    pct = (token_count / model.config.max_position_embeddings) * 100
    print(f"Prompt : '{p}'")
    print(f"Tokens : {token_count} / {model.config.max_position_embeddings} ({pct:.1f}% of context used)")
    print("-" * 60)

distilgpt2 max position embeddings: 1024

=== TOKEN COUNT VS CONTEXT WINDOW ===

Prompt : 'AI'
Tokens : 1 / 1024 (0.1% of context used)
------------------------------------------------------------
Prompt : 'AI is transforming the world'
Tokens : 5 / 1024 (0.5% of context used)
------------------------------------------------------------
Prompt : 'AI is transforming the world in ways we never imagined possible before'
Tokens : 12 / 1024 (1.2% of context used)
------------------------------------------------------------


## 5. Multi-Head Attention
distilgpt2 has multiple attention heads per layer.
Each head learns to attend to different patterns.
- Some heads track grammar (subject → verb)
- Some heads track coreference (pronoun → noun)
- Some heads track position (recent tokens)

In [6]:
print("=== COMPARING ATTENTION HEADS (Layer 0) ===\n")
num_heads = attentions[0].shape[1]
print(f"Number of heads: {num_heads}\n")

# For the last token, show what each head attends to most
last_token_idx = len(tokens) - 1
last_token = tokens[last_token_idx]

print(f"For token '{last_token}', each head attends most to:\n")
for h in range(num_heads):
    head_weights = attentions[0][0][h][last_token_idx]
    max_idx = head_weights.argmax().item()
    max_score = head_weights[max_idx].item()
    print(f"  Head {h}: '{tokens[max_idx]}' (score: {max_score:.3f})")

=== COMPARING ATTENTION HEADS (Layer 0) ===

Number of heads: 12

For token 'Ġtired', each head attends most to:

  Head 0: 'Ġsat' (score: 0.193)
  Head 1: 'Ġtired' (score: 0.993)
  Head 2: 'The' (score: 0.268)
  Head 3: 'Ġtired' (score: 0.999)
  Head 4: 'Ġtired' (score: 0.639)
  Head 5: 'Ġtired' (score: 1.000)
  Head 6: 'The' (score: 0.163)
  Head 7: 'Ġwas' (score: 0.327)
  Head 8: 'The' (score: 0.322)
  Head 9: 'The' (score: 0.384)
  Head 10: 'Ġtired' (score: 0.416)
  Head 11: 'The' (score: 0.379)


## 6. Attention Across Layers
Early layers = local patterns (nearby tokens)
Later layers = global patterns (long-range dependencies)

In [7]:
print("=== ATTENTION ACROSS LAYERS (Head 0, Last Token) ===\n")
num_layers = len(attentions)
print(f"Number of layers: {num_layers}\n")

for l in range(num_layers):
    head_weights = attentions[l][0][0][last_token_idx]
    max_idx = head_weights.argmax().item()
    max_score = head_weights[max_idx].item()
    print(f"Layer {l}: attends most to '{tokens[max_idx]}' (score: {max_score:.3f})")

=== ATTENTION ACROSS LAYERS (Head 0, Last Token) ===

Number of layers: 6

Layer 0: attends most to 'Ġsat' (score: 0.193)
Layer 1: attends most to 'The' (score: 0.315)
Layer 2: attends most to 'The' (score: 0.690)
Layer 3: attends most to 'The' (score: 0.437)
Layer 4: attends most to 'The' (score: 0.612)
Layer 5: attends most to 'The' (score: 0.231)


## 7. Why Context Window Matters for LLM Engineering

Real-world implications:
1. Long documents must be chunked if they exceed context window
2. RAG systems inject only the most relevant chunks
3. Chat history gets truncated when it exceeds the window
4. Cost = proportional to tokens in context (API pricing)
5. Latency = increases with context length

This is why context management is a core skill for LLM engineers.

In [8]:
def count_tokens(text, tokenizer):
    return len(tokenizer.encode(text))

def fits_in_context(text, tokenizer, model, buffer=100):
    count = count_tokens(text, tokenizer)
    max_ctx = model.config.max_position_embeddings
    fits = count < (max_ctx - buffer)
    return fits, count, max_ctx

# Test it
test_text = "AI engineers build systems that scale." * 50
fits, count, max_ctx = fits_in_context(test_text, tokenizer, model)
print(f"Text token count : {count}")
print(f"Context window   : {max_ctx}")
print(f"Fits in context  : {fits}")

Text token count : 350
Context window   : 1024
Fits in context  : True


## 8. Observations & Notes

### What is attention in plain English?
At every token position, the model calculates a score for every
other token — "how relevant is this token to predicting the next one?"
High score = pay more attention. The scores sum to 1 (softmax).

### What did we observe?
- Pronoun "it" attends strongly to "cat" — coreference resolution
- Different heads attend to different tokens — specialization
- Later layers show longer-range dependencies than early layers

### Context window key numbers
- distilgpt2 : 1024 tokens (~750 words)
- GPT-3      : 4096 tokens
- GPT-4      : 128k tokens
- Claude     : 200k tokens

### Why does context window size matter?
Bigger context = model can "remember" more of the conversation.
But bigger context = more computation = higher cost + latency.
LLM engineers must balance what to keep in context vs retrieve via RAG.

### Connection to RAG
RAG exists because context windows have limits.
Instead of stuffing everything in, RAG retrieves only relevant chunks.
This is what you'll build next — the embeddings + retrieval system.

### My questions after this session
(write your own here)
```

---

## Notes for your copy

**1. The attention formula**
```
Attention(Q, K, V) = softmax(QK^T / √d_k) · V